In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

Error importing in API mode: ImportError("dlopen(/opt/anaconda3/envs/fd_library/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): symbol not found in flat namespace '_R_BaseEnv'")
Trying to import in ABI mode.


In [2]:
import pandas as pd

from utils import (
    functional_richness,
    functional_evenness,
    functional_divergence,
    functional_dispersion,
    raos_Q,
)
from utils import euclidean_distance

## Testing on a small dataset

In [3]:
traits = pd.DataFrame(
    [[1, 2], [2, 3], [3, 1], [4, 2]],
    columns=["Trait_1", "Trait_2"],
    index=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
)

abundances = pd.DataFrame(
    [[5, 3, 2, 1], [1, 2, 0, 2]],
    columns=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
    index=["Plot_A", "Plot_B"],
)

In [4]:
FRic = functional_richness(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

distance_matrix_euclidean = euclidean_distance(
    traits, metric="euclidean", standardize_method="z_score"
)
FEve = functional_evenness(
    abundances, distance_matrix_euclidean, relative_abundance=False
)

FDiv = functional_divergence(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

FDis = functional_dispersion(
    abundances, traits, relative_abundance=False, standardize_traits_method="z_score"
)

raos_Q_df = raos_Q(abundances, distance_matrix_euclidean, relative_abundance=False)

python_results_df = (
    FRic.merge(FEve, on="PID")
    .merge(FDiv, on="PID")
    .merge(FDis, on="PID")
    .merge(raos_Q_df, on="PID")
)
display(python_results_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,Plot_A,3.794733,0.734321,0.947928,1.412440,3.371901
1,Plot_B,1.897367,0.989082,0.846568,1.278219,3.264000


In [5]:
r_results_df = None

In [7]:
%%R -i traits,abundances -o r_results_df
library(FD)


trait_mat <- as.matrix(traits)
abun_mat <- as.matrix(abundances)

res <- dbFD(x = trait_mat, a = abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE)

r_results_df <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. The 2 PCoA axes were kept as 'traits'. 


In [8]:
display(r_results_df)

,PID,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ
Plot_A,Plot_A,2.846050,0.734321,0.947928,1.063337,1.264463
Plot_B,Plot_B,1.423025,0.989082,0.846568,1.090310,1.224000


In [10]:
df_merge = python_results_df.merge(r_results_df, on="PID")
df_merge["FRic_ratio"] = df_merge["Functional_Richness"] / df_merge["R_FRic"]
df_merge["FEve_ratio"] = df_merge["Functional_Evenness"] / df_merge["R_FEve"]
df_merge["FDiv_ratio"] = df_merge["Functional_Divergence"] / df_merge["R_FDiv"]
df_merge["FDis_ratio"] = df_merge["Functional_Dispersion"] / df_merge["R_FDis"]
df_merge["RaoQ_ratio"] = df_merge["Raos_Q"] / df_merge["R_RaoQ"]
display(df_merge)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,Plot_A,3.794733,0.734321,0.947928,1.412440,3.371901,2.846050,0.734321,0.947928,1.063337,1.264463,1.333333,1.0,1.0,1.328309,2.666667
1,Plot_B,1.897367,0.989082,0.846568,1.278219,3.264000,1.423025,0.989082,0.846568,1.090310,1.224000,1.333333,1.0,1.0,1.172345,2.666667


## Testing on the birds dataset

In [11]:
bird_loc = pd.read_csv("./data/example/bird/bird_location.csv")
bird_traits = pd.read_csv("./data/example/bird/bird_traits.csv")

bird_loc = bird_loc.set_index("PID")
bird_traits = bird_traits.set_index("Species")

In [24]:
FRic_bird = functional_richness(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)


distance_matrix_euclidean_bird = euclidean_distance(
    bird_traits, metric="euclidean", standardize_method="z_score"
)
FEve_bird = functional_evenness(
    bird_loc,
    distance_matrix_euclidean_bird,
    relative_abundance=False,
    abundance_weighted=True,
)

FDiv_bird = functional_divergence(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)

FDis_bird = functional_dispersion(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits_method="z_score"
)

distance_matrix_euclidean_bird_z_score = euclidean_distance(
    bird_traits, metric="euclidean", standardize_method="z_score"
)
raos_Q_df_bird = raos_Q(
    bird_loc, distance_matrix_euclidean_bird_z_score, relative_abundance=False
)

python_results_df_bird = (
    FRic_bird.merge(FEve_bird, on="PID")
    .merge(FDiv_bird, on="PID")
    .merge(FDis_bird, on="PID")
    .merge(raos_Q_df_bird, on="PID")
)
display(python_results_df_bird)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,elev_250,66.661794,0.656412,0.747405,1.702557,9.191579
1,elev_500,72.128929,0.651052,0.755107,1.741823,9.438064
2,elev_1000,43.756363,0.623858,0.743327,1.568195,7.925136
3,elev_1500,25.703033,0.568285,0.742684,1.474851,7.206896
4,elev_2000,7.797544,0.605025,0.730325,1.247874,4.750789
5,elev_2500,7.111826,0.631438,0.703676,1.268754,5.043357
6,elev_3000,6.812401,0.616293,0.700852,1.334470,5.383485
7,elev_3500,1.441212,0.592614,0.671813,1.348770,5.876478


In [13]:
r_results_df_bird = None

In [17]:
%%R -i bird_loc,bird_traits -o r_results_df_bird
library(FD)
bird_trait_mat <- as.matrix(bird_traits)
bird_abun_mat <- as.matrix(bird_loc)

res <- dbFD(x = bird_trait_mat, a = bird_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_bird <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. All 4 PCoA axes were kept as 'traits'. 


In [27]:
merge_df = python_results_df_bird.merge(r_results_df_bird, on="PID")

merge_df["FRic_ratio"] = merge_df["Functional_Richness"] / merge_df["R_FRic"]
merge_df["FEve_ratio"] = merge_df["Functional_Evenness"] / merge_df["R_FEve"]
merge_df["FDiv_ratio"] = merge_df["Functional_Divergence"] / merge_df["R_FDiv"]
merge_df["FDis_ratio"] = merge_df["Functional_Dispersion"] / merge_df["R_FDis"]
merge_df["RaoQ_ratio"] = merge_df["Raos_Q"] / merge_df["R_RaoQ"]

display(merge_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,elev_250,66.661794,0.656412,0.747405,1.702557,9.191579,66.048816,0.656412,0.747405,1.698629,4.574611,1.009281,1.0,1.0,1.002312,2.009259
1,elev_500,72.128929,0.651052,0.755107,1.741823,9.438064,71.465678,0.651052,0.755107,1.737805,4.697285,1.009281,1.0,1.0,1.002312,2.009259
2,elev_1000,43.756363,0.623858,0.743327,1.568195,7.925136,43.354008,0.623858,0.743327,1.564577,3.944307,1.009281,1.0,1.0,1.002312,2.009259
3,elev_1500,25.703033,0.568285,0.742684,1.474851,7.206896,25.466685,0.568285,0.742684,1.471448,3.586842,1.009281,1.0,1.0,1.002312,2.009259
4,elev_2000,7.797544,0.605025,0.730325,1.247874,4.750789,7.725843,0.605025,0.730325,1.244996,2.364448,1.009281,1.0,1.0,1.002312,2.009259
5,elev_2500,7.111826,0.631438,0.703676,1.268754,5.043357,7.046431,0.631438,0.703676,1.265827,2.510058,1.009281,1.0,1.0,1.002312,2.009259
6,elev_3000,6.812401,0.616293,0.700852,1.334470,5.383485,6.749758,0.616293,0.700852,1.331392,2.679338,1.009281,1.0,1.0,1.002312,2.009259
7,elev_3500,1.441212,0.592614,0.671813,1.348770,5.876478,1.427960,0.592614,0.671813,1.345659,2.924699,1.009281,1.0,1.0,1.002312,2.009259
